In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/malaria_disease.csv")

#### drop leakage features

In [2]:
leakage_features = [
    "rdt_positive",
    "microscopy_positive",
    "pcr_positive",
    "parasitemia_percent"
]

df_diag = df.drop(columns=leakage_features)

#### remove non predictive column

In [3]:
df_diag = df_diag.drop(columns=["patient_id"])

#### check the remaining columns

In [4]:
df_diag.columns

Index(['age', 'sex', 'pregnant', 'location', 'travel_history_endemic_area',
       'bed_net_use', 'irs_spraying', 'previous_malaria_episodes', 'fever',
       'chills_rigors', 'headache', 'night_sweats', 'fatigue_malaise',
       'nausea_vomiting', 'diarrhea', 'cough', 'abdominal_pain', 'jaundice',
       'altered_consciousness', 'seizures', 'hemoglobin_g_dl',
       'platelets_x10e9_l', 'wbc_x10e9_l', 'glucose_mg_dl', 'creatinine_mg_dl',
       'bilirubin_mg_dl', 'lactate_mmol_l', 'plasmodium_species', 'severity',
       'treatment_received', 'outcome', 'diagnosis'],
      dtype='str')

#### encode the target

In [5]:
from sklearn.preprocessing import LabelEncoder

le_diagnosis = LabelEncoder()

df_diag["diagnosis"] = le_diagnosis.fit_transform(
    df_diag["diagnosis"]
)

#### verify the target

In [6]:
dict(
    zip(
        le_diagnosis.classes_,
        le_diagnosis.transform(
            le_diagnosis.classes_
        )
    )
)

{'Malaria': np.int64(0), 'No Malaria': np.int64(1)}

#### create malaria severity dataset
we only want malaria positive patients 

In [7]:
severity_df = df[
    df["diagnosis"] == "Malaria"
].copy()

In [8]:
severity_df = severity_df.drop(
    columns=[
        "patient_id",
        "diagnosis"
    ]
)

#### remove sever leakage
For severity prediction, we should investigate whether some lab-confirmation variables still create unrealistic leakage. For now, remove the same four:

In [9]:
severity_df = severity_df.drop(
    columns=[
        "rdt_positive",
        "microscopy_positive",
        "pcr_positive",
        "parasitemia_percent"
    ]
)

#### Remove "No Malaria" Severity Class

In [10]:
severity_df = severity_df[
    severity_df["severity"] != "No Malaria"
]

#### save processed data

In [11]:
df_diag.to_csv(
    "../data/processed/diagnosis_dataset.csv",
    index=False
)

severity_df.to_csv(
    "../data/processed/severity_dataset.csv",
    index=False
)

In [14]:
df_diag.shape



(5000, 32)

In [15]:
severity_df.shape

(1528, 31)

In [16]:
severity_df["severity"].value_counts()

severity
Uncomplicated    796
Moderate         650
Severe            82
Name: count, dtype: int64

In [17]:
df_diag.dtypes

age                              int64
sex                                str
pregnant                          bool
location                           str
travel_history_endemic_area       bool
bed_net_use                       bool
irs_spraying                      bool
previous_malaria_episodes        int64
fever                             bool
chills_rigors                     bool
headache                          bool
night_sweats                      bool
fatigue_malaise                   bool
nausea_vomiting                   bool
diarrhea                          bool
cough                             bool
abdominal_pain                    bool
jaundice                          bool
altered_consciousness             bool
seizures                          bool
hemoglobin_g_dl                float64
platelets_x10e9_l                int64
wbc_x10e9_l                    float64
glucose_mg_dl                    int64
creatinine_mg_dl               float64
bilirubin_mg_dl          

#### create final diagonistic dataset

In [18]:
diagnosis_drop_cols = [
    "severity",
    "treatment_received",
    "outcome",
    "plasmodium_species"
]

df_diag_final = df_diag.drop(columns=diagnosis_drop_cols)

df_diag_final.shape

(5000, 28)

#### save final dataset

In [19]:
df_diag_final.to_csv(
    "../data/processed/diagnosis_final.csv",
    index=False
)